# 03 — Exploratory Data Analysis
**Saudi Tourism Intelligence Project**

5 Key Analyses:
1. Tourist Trends
2. Seasonality
3. Spending Analysis
4. Overnight Stays
5. Carbon Impact Index

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

base = '/content/drive/MyDrive/Saudi_Tourism_Project/data/clean'
tourists = pd.read_csv(f'{base}/01_Tourists_CLEAN.csv')
overnight = pd.read_csv(f'{base}/02_Overnight_CLEAN.csv')
kpi = pd.read_csv(f'{base}/03_KPI_CLEAN.csv')
exp = pd.read_csv(f'{base}/04_Expenditure_CLEAN.csv')
print('Data loaded ✅')

## 1. Tourist Trends 2015-2024

In [ ]:
annual = tourists[tourists['Frequency']=='Annual'].copy()
annual['Period'] = annual['Period'].astype(int)
total = annual.groupby('Period')['Tourists'].sum()

fig = px.line(total.reset_index(), x='Period', y='Tourists',
              title='Total Tourists 2015-2024',
              markers=True)
fig.show()
print(total.to_string())

## 2. Seasonality Analysis

In [ ]:
monthly = tourists[tourists['Frequency']=='Monthly'].copy()
monthly['Month_Num'] = monthly['Period'].str[-2:].astype(int)
by_month = monthly.groupby('Month_Num')['Tourists'].sum().reset_index()

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
by_month['Month_Name'] = by_month['Month_Num'].map(month_names)

fig = px.bar(by_month, x='Month_Name', y='Tourists',
             title='Seasonality — Tourists by Month')
fig.show()

## 3. Spending Analysis

In [ ]:
kpi['Year'] = kpi['Month'].str[:4].astype(int)
spending = kpi.groupby(['Year','Type of tourist'])['Average Spending per Trip'].mean().reset_index()

fig = px.line(spending, x='Year', y='Average Spending per Trip',
              color='Type of tourist',
              title='Avg Spending per Trip 2015-2024',
              markers=True)
fig.show()

## 4. Overnight Stays

In [ ]:
annual_o = overnight[overnight['Frequency']=='Annual'].copy()
annual_o['Period'] = annual_o['Period'].astype(int)
overnight_trend = annual_o.groupby(['Period','Type of tourist'])['Overnight Stays'].sum().reset_index()

fig = px.bar(overnight_trend, x='Period', y='Overnight Stays',
             color='Type of tourist',
             title='Overnight Stays 2015-2024',
             barmode='group')
fig.show()

## 5. Carbon Impact Index

In [ ]:
CARBON_INBOUND = 89.0
CARBON_DOMESTIC = 34.0

annual_o['Carbon_MT'] = annual_o.apply(
    lambda r: r['Overnight Stays'] * 1000 * CARBON_INBOUND / 1e9
    if r['Type of tourist'] == 'Inbound'
    else r['Overnight Stays'] * 1000 * CARBON_DOMESTIC / 1e9, axis=1)

carbon_total = annual_o.groupby('Period')['Carbon_MT'].sum().reset_index()

fig = px.area(carbon_total, x='Period', y='Carbon_MT',
              title='Carbon Impact Index (MegaTons CO2)',
              color_discrete_sequence=['#00B4B4'])
fig.show()
carbon_total.to_csv(f'{base}/05_Carbon_Impact.csv', index=False)
print('Carbon Impact saved ✅')